# 01: Data Preprocessing

Prepares the NCRP prison records dataset (~13.9M rows, 44 states) for causal analysis of prison education on recidivism. Builds the outcome variable (`within_2_yrs`), filters invalid records, and merges state-level education policy scores.

**Input:** `data/raw/dataset.csv` | **Output:** `data/processed/clean_recidivism.csv`

## 1. Imports

In [1]:
import os
import numpy as np
import pandas as pd
import gdown

## 2. Load Data

In [2]:
file_id = "1jkBkjOsdr-0hhUgj3p2J7pk1Welkk7A1"
raw_path = "../data/raw/dataset.csv"

os.makedirs(os.path.dirname(raw_path), exist_ok=True)

if not os.path.exists(raw_path):
    gdown.download(id=file_id, output=raw_path, quiet=False)

dtypes = {
    "ABT_INMATE_ID": "str",
    "SEX": "int8",
    "ADMTYPE": "int8",
    "OFFGENERAL": "int8",
    "EDUCATION": "int8",
    "SENTLGTH": "int8",
    "OFFDETAIL": "int8",
    "RACE": "int8",
    "AGEADMIT": "int8",
    "AGERELEASE": "int8",
    "TIMESRVD": "int8",
    "RELTYPE": "int8",
    "STATE": "int8",
    "ADMITYR": "int16",
    "RELEASEYR": "int16",
}

df = pd.read_csv(raw_path, sep="\t", dtype=dtypes, low_memory=False)
print(
    f"Loaded: {df.shape[0]:,} rows - {df.shape[1]} cols | {df.memory_usage(deep=True).sum()/1e9:.2f} GB"
)

Downloading...
From (original): https://drive.google.com/uc?id=1jkBkjOsdr-0hhUgj3p2J7pk1Welkk7A1
From (redirected): https://drive.google.com/uc?id=1jkBkjOsdr-0hhUgj3p2J7pk1Welkk7A1&confirm=t&uuid=ce358256-5b61-4650-9aea-7223f01fb3de
To: /Users/temesmacbookair/Development/ids701_spds/Recidivsm_CleanRepo/data/raw/dataset.csv
100%|██████████| 976M/976M [00:25<00:00, 37.9MB/s] 


Loaded: 13,897,856 rows - 18 cols | 1.50 GB


## 3. Sanity Check

Confirm shape, memory, and sentinel value counts before touching anything.

In [3]:
print(f"Shape: {df.shape} | Memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB\n")

sentinels = {
    "RELEASEYR = 9999  (still incarcerated)": (df["RELEASEYR"] == 9999).sum(),
    "ADMITYR   = 9999  (missing)": (df["ADMITYR"] == 9999).sum(),
    "RACE      = 9     (missing)": (df["RACE"] == 9).sum(),
    "EDUCATION = 9     (entirely missing)": (df["EDUCATION"] == 9).sum(),
    "RELTYPE   = 9     (missing)": (df["RELTYPE"] == 9).sum(),
}
for label, count in sentinels.items():
    print(f"  {label}: {count:>9,}  ({count/len(df):.1%})")

display(df.head(3))

Shape: (13897856, 18) | Memory: 1.50 GB

  RELEASEYR = 9999  (still incarcerated):   996,467  (7.2%)
  ADMITYR   = 9999  (missing):       658  (0.0%)
  RACE      = 9     (missing): 1,318,720  (9.5%)
  EDUCATION = 9     (entirely missing): 13,897,856  (100.0%)
  RELTYPE   = 9     (missing): 1,646,999  (11.9%)


,ABT_INMATE_ID,SEX,ADMTYPE,OFFGENERAL,EDUCATION,ADMITYR,RELEASEYR,MAND_PRISREL_YEAR,PROJ_PRISREL_YEAR,PARELIG_YEAR,SENTLGTH,OFFDETAIL,RACE,AGEADMIT,AGERELEASE,TIMESRVD,RELTYPE,STATE
0,A012021000000090128,1,1,2,9,2013,2013,9999,9999,9999,4,8,1,5,5,0,3,1
1,A012021000000110168,1,1,1,9,2004,9999,9999,9999,9999,4,3,1,5,9,9,9,1
2,A012021000000090187,1,1,2,9,2009,2009,9999,9999,9999,0,11,9,1,1,0,1,1


## 4. Drop Dead Columns

`EDUCATION` is entirely sentinel (9 = missing for all 13.9M rows — unusable).  
`MAND_PRISREL_YEAR`, `PROJ_PRISREL_YEAR`, `PARELIG_YEAR` are theoretical legal dates, nearly all 9999. None are used in any analysis.

In [4]:
dead_cols = ["EDUCATION", "MAND_PRISREL_YEAR", "PROJ_PRISREL_YEAR", "PARELIG_YEAR"]
df = df.drop(columns=dead_cols)
print(f"Dropped: {dead_cols}")
print(f"Shape: {df.shape} | Memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB")

Dropped: ['EDUCATION', 'MAND_PRISREL_YEAR', 'PROJ_PRISREL_YEAR', 'PARELIG_YEAR']
Shape: (13897856, 14) | Memory: 1.15 GB


## 5. Decode Sentinel Values

NCRP encodes missing as `9`, `99`, or `9999` depending on the column. Convert to `NaN` before any derived variables are built.

In [5]:
df["RACE"] = df["RACE"].replace(9, np.nan)
df["AGEADMIT"] = df["AGEADMIT"].replace(9, np.nan)
df["AGERELEASE"] = df["AGERELEASE"].replace(9, np.nan)
df["OFFDETAIL"] = df["OFFDETAIL"].replace(99, np.nan)
df["SENTLGTH"] = df["SENTLGTH"].replace(9, np.nan)
df["RELTYPE"] = df["RELTYPE"].replace(9, np.nan)

missing = (df.isna().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0].rename("% missing")
print(missing.to_string())

RELTYPE       11.850742
RACE           9.488658
AGERELEASE     7.511900
SENTLGTH       0.784063
OFFDETAIL      0.500322
AGEADMIT       0.372604


## 6. Remove Duplicates

In [6]:
n_before = len(df)
df = df.drop_duplicates(subset=["ABT_INMATE_ID", "ADMITYR", "RELEASEYR"], keep="first")
print(f"Removed {n_before - len(df):,} duplicates - {len(df):,} rows remain")

Removed 213,506 duplicates - 13,684,350 rows remain


## 7. Map State Codes to Names

In [7]:
state_map = {
    1: "Alabama",
    2: "Alaska",
    4: "Arizona",
    5: "Arkansas",
    6: "California",
    8: "Colorado",
    9: "Connecticut",
    10: "Delaware",
    11: "District of Columbia",
    12: "Florida",
    13: "Georgia",
    15: "Hawaii",
    16: "Idaho",
    17: "Illinois",
    18: "Indiana",
    19: "Iowa",
    20: "Kansas",
    21: "Kentucky",
    22: "Louisiana",
    23: "Maine",
    24: "Maryland",
    25: "Massachusetts",
    26: "Michigan",
    27: "Minnesota",
    28: "Mississippi",
    29: "Missouri",
    30: "Montana",
    31: "Nebraska",
    32: "Nevada",
    33: "New Hampshire",
    34: "New Jersey",
    35: "New Mexico",
    36: "New York",
    37: "North Carolina",
    38: "North Dakota",
    39: "Ohio",
    40: "Oklahoma",
    41: "Oregon",
    42: "Pennsylvania",
    44: "Rhode Island",
    45: "South Carolina",
    46: "South Dakota",
    47: "Tennessee",
    48: "Texas",
    49: "Utah",
    50: "Vermont",
    51: "Virginia",
    53: "Washington",
    54: "West Virginia",
    55: "Wisconsin",
}

df["STATE_NAME"] = df["STATE"].map(state_map)
print(
    f"{df['STATE_NAME'].nunique()} states mapped | {df['STATE_NAME'].isna().sum():,} unmapped rows"
)

43 states mapped | 15,486 unmapped rows


## 8. Pre-Outcome Filters

Remove only the records that make `within_2_yrs` unmeasurable. All other filters happen **after** the outcome is built so that future admissions (2019–2020) remain visible for the sort-shift.

| Filter | Reason |
|--------|--------|
| `RELEASEYR = 9999` | Still incarcerated — no release date to measure from |
| `ADMITYR = 9999` | Missing admit year — can't place in timeline |

In [8]:
n = len(df)

df = df[df["RELEASEYR"] != 9999]
print(f"Remove still-incarcerated (RELEASEYR=9999): {n - len(df):>8,} removed  →  {len(df):,} remain"); n = len(df)

df = df[df["ADMITYR"] != 9999]
print(f"Remove missing admit year  (ADMITYR=9999):  {n - len(df):>8,} removed  →  {len(df):,} remain")

Remove still-incarcerated (RELEASEYR=9999):  996,467 removed  →  12,687,883 remain
Remove missing admit year  (ADMITYR=9999):       346 removed  →  12,687,537 remain


## 9. Build Recidivism Outcome

`within_2_yrs = 1` if the inmate is readmitted within 2 years of release, else 0.

Built **before** the analysis sample filters so that re-admissions in 2019–2020 are still visible when measuring recidivism for 2017–2018 releases. Building after `RELEASEYR ≤ 2018` would silently zero out recidivism for the most recent cohorts (right-censoring).

In [9]:
df = df.sort_values(["ABT_INMATE_ID", "ADMITYR"])

df["NEXT_ADMITYR"] = df.groupby("ABT_INMATE_ID")["ADMITYR"].shift(-1)

df["within_2_yrs"] = (
    (df["NEXT_ADMITYR"] > df["RELEASEYR"]) &
    (df["NEXT_ADMITYR"] <= df["RELEASEYR"] + 2)
).astype(int)

df.drop(columns=["NEXT_ADMITYR"], inplace=True)

print(f"Recidivism rate (pre-analysis filter): {df['within_2_yrs'].mean():.1%}")
print(f"Shape: {df.shape}")

Recidivism rate (pre-analysis filter): 24.5%
Shape: (12687537, 16)


## 10. Analysis Sample Filters

Apply remaining restrictions now that `within_2_yrs` is correctly computed.

| Filter | Reason |
|--------|--------|
| `RELTYPE ∈ {5, 6}` | Death or escape — cannot recidivate |
| States missing pre-2010 data | Need sufficient pre-treatment window for DiD |
| New Mexico, West Virginia | Gaps in 2011–2018 coverage |
| `RELEASEYR > 2018` | 2-year window requires data through 2020; dataset ends there |

In [10]:
n = len(df)

df = df[~df["RELTYPE"].isin([5, 6])]
print(f"Remove death / escape:            {n - len(df):>8,} removed  →  {len(df):,} remain"); n = len(df)

valid_states = df.groupby("STATE_NAME")["RELEASEYR"].min().loc[lambda x: x <= 2010].index
df = df[df["STATE_NAME"].isin(valid_states)]
print(f"Keep states with data from 2010:  {n - len(df):>8,} removed  →  {len(df):,} remain"); n = len(df)

df = df[~df["STATE_NAME"].isin(["New Mexico", "West Virginia"])]
print(f"Remove NM, WV (coverage gaps):    {n - len(df):>8,} removed  →  {len(df):,} remain"); n = len(df)

df = df[df["RELEASEYR"] <= 2018]
print(f"Cap at RELEASEYR <= 2018:         {n - len(df):>8,} removed  →  {len(df):,} remain")

df["tot_arrest_counts"] = df.groupby("ABT_INMATE_ID")["ABT_INMATE_ID"].transform("count")

print(f"\nFinal sample:    {len(df):,} rows")
print(f"Recidivism rate: {df['within_2_yrs'].mean():.1%}")

Remove death / escape:                   0 removed  →  12,687,537 remain
Keep states with data from 2010:   266,741 removed  →  12,420,796 remain
Remove NM, WV (coverage gaps):      48,755 removed  →  12,372,041 remain
Cap at RELEASEYR <= 2018:          824,056 removed  →  11,547,985 remain

Final sample:    11,547,985 rows
Recidivism rate: 26.3%


## 10. Add Policy Adoption Variables

Merge state-level prison education policy measures onto the cleaned NCRP release records. These variables are fixed at the state level and used later in the regression notebooks. This notebook only constructs and validates them; model specifications are handled separately.

In [11]:
policy_data = [
    ["Ohio", 1, 1, 0.5, 1, 1, 1, 1, 6.5],
    ["California", 0.75, 0.75, 0.75, 1, 1, 1, 1, 6.25],
    ["Wyoming", 1, 1, 1, 1, 1, 0, 1, 6.0],
    ["Iowa", 1, 1, 1, 0.75, 1, 0, 1, 5.75],
    ["Rhode Island", 1, 1, 1, 0.5, 0, 1, 1, 5.5],
    ["Vermont", 1, 1, 1, 0.5, 0, 1, 1, 5.5],
    ["Illinois", 0.75, 0.25, 0.5, 0.75, 1, 1, 1, 5.25],
    ["New Hampshire", 0.25, 1, 1, 1, 0, 1, 1, 5.25],
    ["New York", 0.75, 0.5, 0.75, 1, 0, 1, 1, 5.0],
    ["North Dakota", 1, 1, 1, 1, 0, 0, 1, 5.0],
    ["New Jersey", 0.5, 1, 0.75, 0.5, 1, 0, 1, 4.75],
    ["West Virginia", 0.75, 0.5, 0.5, 1, 0, 1, 1, 4.75],
    ["New Mexico", 0.5, 1, 0, 1, 1, 0, 1, 4.5],
    ["South Carolina", 1, 0, 0.5, 0, 1, 1, 1, 4.5],
    ["Texas", 0.5, 0.5, 0.25, 0.25, 1, 1, 1, 4.5],
    ["Idaho", 0.5, 1, 0.75, 0, 0, 1, 1, 4.25],
    ["Tennessee", 0.75, 0.25, 0.75, 0.5, 0, 1, 1, 4.25],
    ["Connecticut", 0.5, 0.75, 0, 0.75, 0, 1, 1, 4.0],
    ["Maine", 1, 1, 1, 1, 0, 0, 0, 4.0],
    ["Oregon", 0.5, 0.5, 0.75, 0.25, 1, 0, 1, 4.0],
    ["Utah", 1, 1, 1, 0, 0, 0, 1, 4.0],
    ["Arkansas", 0.25, 0.5, 0, 0, 1, 1, 1, 3.75],
    ["Washington", 0, 1, 0.75, 1, 0, 0, 1, 3.75],
    ["Michigan", 0.75, 0.5, 0.25, 0.25, 1, 1, 0, 3.75],
    ["Mississippi", 0.5, 0.5, 0.75, 0.75, 0, 0, 1, 3.5],
    ["Colorado", 0.5, 0.5, 0.5, 0.75, 0, 0, 1, 3.25],
    ["Delaware", 0.75, 1, 0.25, 0.25, 0, 0, 1, 3.25],
    ["Indiana", 0.5, 0.5, 0.75, 0.5, 0, 0, 1, 3.25],
    ["Nebraska", 0.25, 0.25, 0.25, 0.5, 0, 1, 1, 3.25],
    ["Hawaii", 1, 1, 1, 0, 0, 0, 0, 3.0],
    ["Kansas", 1, 0, 1, 0, 0, 0, 1, 3.0],
    ["Louisiana", 0, 1, 0.25, 0.75, 0, 0, 1, 3.0],
    ["Massachusetts", 0, 0, 0.25, 0.75, 0, 1, 1, 3.0],
    ["Minnesota", 1, 1, 0, 0, 0, 1, 0, 3.0],
    ["Pennsylvania", 0.5, 0.5, 0.75, 0.25, 0, 1, 0, 3.0],
    ["Arizona", 0, 0, 0.5, 0.25, 1, 0, 1, 2.75],
    ["Kentucky", 0.25, 0.5, 0.5, 0.5, 0, 0, 1, 2.75],
    ["North Carolina", 0, 0, 0.25, 0.5, 1, 0, 1, 2.75],
    ["Oklahoma", 0.25, 0.25, 0.25, 0.75, 0, 0, 1, 2.5],
    ["Wisconsin", 0.25, 0.25, 0.5, 0.5, 0, 1, 0, 2.5],
    ["Maryland", 0.25, 0.25, 0.5, 0.25, 0, 0, 1, 2.25],
    ["Virginia", 0.25, 0.25, 0.5, 0.25, 0, 0, 1, 2.25],
    ["Florida", 0, 0, 0, 0, 1, 0, 1, 2.0],
    ["Georgia", 0.5, 0.25, 0, 0.25, 0, 0, 1, 2.0],
    ["Nevada", 0, 0, 0.25, 0.75, 0, 0, 1, 2.0],
    ["South Dakota", 1, 1, 0, 0, 0, 0, 0, 2.0],
    ["Alabama", 0, 0, 0, 0.5, 0, 0, 1, 1.5],
    ["Alaska", 0, 0, 0, 0, 0, 0, 1, 1.0],
    ["Missouri", 0.5, 0.25, 0, 0.25, 0, 0, 0, 1.0],
    ["Montana", 0, 0, 0.25, 0.75, 0, 0, 0, 1.0],
]

policy_df = pd.DataFrame(
    policy_data,
    columns=[
        "STATE_NAME",
        "ABE_Literacy",
        "Secondary",
        "Vocational",
        "College",
        "Automatic_Enrollment",
        "School_District",
        "Sentence_Reduction",
        "Total_Score",
    ],
)

## 11. Merge Policy Data

Attach each state's policy measures to every release record.

In [12]:
policy_cols = [
    "ABE_Literacy",
    "Secondary",
    "Vocational",
    "College",
    "Automatic_Enrollment",
    "School_District",
    "Sentence_Reduction",
    "Total_Score",
]

# Make this cell safe to rerun after policy columns have already been merged.
df = df.drop(columns=[col for col in policy_cols if col in df.columns])

n = len(df)
df = df.merge(policy_df, on="STATE_NAME", how="left", validate="many_to_one")

print(f"Rows before merge: {n:,}")
print(f"Rows after merge:  {len(df):,}")

Rows before merge: 11,547,985
Rows after merge:  11,547,985


## 12. Validate Policy Merge

Confirm that all retained states received policy values and inspect the resulting score distribution.

In [13]:
unmatched_states = sorted(
    df.loc[df["Total_Score"].isna(), "STATE_NAME"].dropna().unique()
)

print(f"States in cleaned data: {df['STATE_NAME'].nunique():,}")
print(f"States in policy table: {policy_df['STATE_NAME'].nunique():,}")
print(f"Unmatched states:       {unmatched_states}")

missing_policy = df[policy_cols].isna().sum()
missing_policy[missing_policy > 0]

States in cleaned data: 34
States in policy table: 50
Unmatched states:       ['District of Columbia']


ABE_Literacy            45117
Secondary               45117
Vocational              45117
College                 45117
Automatic_Enrollment    45117
School_District         45117
Sentence_Reduction      45117
Total_Score             45117
dtype: int64

In [14]:
df[["STATE_NAME", "Total_Score"]].drop_duplicates().sort_values(
    "Total_Score", ascending=False
).head(20)

,STATE_NAME,Total_Score
8638488,Ohio,6.50
478741,California,6.25
5749891,Iowa,5.75
9461523,Rhode Island,5.50
4680217,Illinois,5.25
8619374,North Dakota,5.00
7501671,New York,5.00
7315395,New Jersey,4.75
10007347,Texas,4.50
9514713,South Carolina,4.50


## 13. Build Policy Tier

Create a coarse low / medium / high policy score category for optional descriptive analysis.

In [15]:
df["policy_tier"] = pd.qcut(df["Total_Score"], 3, labels=["Low", "Medium", "High"])

df["policy_tier"].value_counts(dropna=False)

policy_tier
Low       4042482
Medium    3775249
High      3685137
NaN         45117
Name: count, dtype: int64

## 14. Export Processed Data

Save the cleaned, analysis-ready dataset for the regression notebooks.

In [16]:
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/clean_recidivism.csv", index=False)

print(f"Saved rows:    {len(df):,}")
print(f"Saved columns: {df.shape[1]:,}")

Saved rows:    11,547,985
Saved columns: 26
